# 2. Gold-set annotation

| | |
|---|---|
| **Stage** | 2 of 9 |
| **Purpose** | Build the hand-labelled gold set: load the aspect taxonomy, define which aspects apply to which game, pre-label with a keyword lexicon, draw 100 sentences per game, review them in an interactive tool, validate, fix genre cross-talk and write the combined gold file. Part 5 (cleaning and sentence splitting) is only needed for raw data that has not been processed. |
| **Inputs** | `scripts/aspect_taxonomy.py`<br>`data/processed/{game}_sentences_weak.parquet`<br>`data/annotation/royalmatch_annotation_sample_FINAL.csv (earlier 1,000-sentence Royal Match sample)` |
| **Outputs** | `data/annotation_v2/{game}_annotation_sample_100.csv`<br>`data/annotation_v2/{game}_annotation_sample_LABELED.csv`<br>`data/annotation_v2/gold_set_combined.csv` |
| **Runtime** | CPU. Interactive (ipywidgets); the manual review is the long part. |
| **Depends on** | Stage 1 (the saved review snapshot, see `docs/DATA.md`) |
| **Configuration** | Set `PROJECT_ROOT` in the first code cell if your Drive folder is not `MyDrive/GameSense`. |

Part 1: Environment setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_ROOT = '/content/drive/MyDrive/GameSense'
for d in ['data/raw', 'data/processed', 'data/manifest', 'data/annotation',
          'data/annotation_v2', 'scripts', 'models', 'outputs']:
    os.makedirs(f'{PROJECT_ROOT}/{d}', exist_ok=True)

%pip install -q -U transformers accelerate spacy pyarrow scikit-learn ipywidgets

import torch
print("GPU available:", torch.cuda.is_available())

Part 2: Load the aspect taxonomy (single source of truth)

In [ ]:
import sys, importlib
sys.path.insert(0, f'{PROJECT_ROOT}/scripts')
if 'aspect_taxonomy' in sys.modules:
    importlib.reload(sys.modules['aspect_taxonomy'])
from aspect_taxonomy import ASPECT_TAXONOMY, ASPECT_IDS

missing = [a for a in ASPECT_IDS if "description" not in ASPECT_TAXONOMY[a]]
assert not missing, f"These aspects have no description: {missing}"
print(f"Taxonomy loaded: {len(ASPECT_IDS)} aspects ✅")

Part 3: Game-aspect applicability mapping

In [ ]:
COMMON_ASPECTS = [
    "performance_stability", "monetization_value", "advertisement",
    "level_difficulty_balance", "content_progression", "paywall",
    "account_support",
]

GENRE_ASPECTS_BY_GAME = {
    "royalmatch": [],
    "genshin":    ["gacha_rates", "character_operator_balance"],
    "mlbb":       ["character_operator_balance", "matchmaking", "cheating"],
    "pubgm":      ["cheating", "matchmaking"],
    "arknights":  ["gacha_rates", "character_operator_balance", "grind_progression"],
}

def get_applicable_aspects(slug):
    return set(COMMON_ASPECTS) | set(GENRE_ASPECTS_BY_GAME.get(slug, []))

Part 4: Lexicon weak labels (pre-annotation and review ordering)

In [ ]:
import re, ast

def match_aspects(sentence):
    s = sentence.lower()
    hits = []
    for aspect_id, meta in ASPECT_TAXONOMY.items():
        for kw in meta.get("keywords", []):
            if kw in s:
                hits.append(aspect_id)
                break
    return hits

def weak_sentiment(sentence):
    NEGATORS = {"not","no","never","n't","cant","can't","cannot","doesnt","doesn't","wont","won't"}
    POS_WORDS = {"good","great","love","loved","fun","amazing","awesome","enjoy","best","smooth","easy","worth"}
    NEG_WORDS = {"bad","terrible","awful","hate","hated","annoying","frustrating","worst","broken","unfair","ripoff","greedy","boring","expensive"}
    tokens = re.findall(r"[a-z']+", sentence.lower())
    score, negate = 0, False
    for tok in tokens:
        if tok in NEGATORS:
            negate = True; continue
        if tok in POS_WORDS: score += -1 if negate else 1
        elif tok in NEG_WORDS: score += 1 if negate else -1
        if tok not in NEGATORS: negate = False
    return "positive" if score > 0 else "negative" if score < 0 else "neutral"

Part 5: Sentence splitting and cleaning (only needed if the raw review data has not been processed yet; skip otherwise)

In [ ]:
import spacy
import pandas as pd

def clean_fragment(text):
    return re.sub(r"\s+", " ", text).strip()

def segment_reviews(df, text_col="content_clean"):
    nlp = spacy.blank("en")
    nlp.add_pipe("sentencizer")
    records = []
    texts = df[text_col].fillna("").astype(str).tolist()
    for doc, (_, row) in zip(nlp.pipe(texts, batch_size=256), df.iterrows()):
        sents = [clean_fragment(s.text) for s in doc.sents]
        sents = [s for s in sents if len(s) >= 3]
        for i, s in enumerate(sents):
            records.append({
                "reviewId": row["reviewId"], "sentence_idx": i, "sentence_text": s,
                "review_score": row.get("score"),
                "review_overall_sentiment_weak": row.get("overall_sentiment_weak"),
                "reviewCreatedVersion": row.get("reviewCreatedVersion"),
                "at": row.get("at"),
            })
    return pd.DataFrame.from_records(records)

def basic_clean_text(text):
    if pd.isna(text): return ""
    text = str(text)
    text = re.sub(r"http\S+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def score_to_weak_sentiment(score):
    return "negative" if score <= 2 else ("neutral" if score == 3 else "positive")

def clean_raw_reviews(df, min_chars=3):
    df = df.copy()
    df["content_clean"] = df["content"].apply(basic_clean_text)
    df["overall_sentiment_weak"] = df["score"].apply(score_to_weak_sentiment)
    before = len(df)
    df = df[df["content_clean"].str.len() >= min_chars]
    df = df.drop_duplicates(subset=["reviewId"])
    print(f'    Cleaning: {before} -> {len(df)} reviews')
    return df

Part 6: Sample size (500 sentences in total, allocated equally: 100 per game)

In [ ]:
# 100 sentences for each of the five games, 500 in total
game_slugs = ["royalmatch", "genshin", "mlbb", "pubgm", "arknights"]
QUOTA_PER_GAME = 100

allocation = {slug: QUOTA_PER_GAME for slug in game_slugs}
print("Equal allocation across the five games:", allocation, " total:", sum(allocation.values()))

Part 7: Generate the annotation samples (Royal Match is processed too: 100 sentences are drawn from its existing 1,000-sentence sample)

In [ ]:
def generate_annotation_sample(slug, quota):
    weak_path = f'{PROJECT_ROOT}/data/processed/{slug}_sentences_weak.parquet'
    old_final_1000_path = f'{PROJECT_ROOT}/data/annotation/{slug}_annotation_sample_FINAL.csv'

    if os.path.exists(old_final_1000_path):
        df = pd.read_csv(old_final_1000_path)
        df['weak_aspects'] = df['weak_aspects'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
    elif os.path.exists(weak_path):
        df = pd.read_parquet(weak_path)
        df['sentence_uid'] = df['reviewId'].astype(str) + "_" + df['sentence_idx'].astype(str)
    else:
        raise FileNotFoundError(f'No sentence data found for {slug}; run Part 5 (cleaning and sentence splitting) first')

    applicable = get_applicable_aspects(slug)
    min_per_aspect = max(3, quota // (len(applicable) * 2))
    chosen, parts = set(), []
    for aspect_id in applicable:
        pool = df[df['weak_aspects'].apply(lambda x: aspect_id in x)]
        pool = pool[~pool['sentence_uid'].isin(chosen)]
        take = pool.sample(n=min(min_per_aspect, len(pool)), random_state=42) if len(pool) else pool
        chosen.update(take['sentence_uid']); parts.append(take)

    rest = df[~df['sentence_uid'].isin(chosen)]
    top_up = rest.sample(n=min(max(0, quota - len(chosen)), len(rest)), random_state=42)
    parts.append(top_up)

    final = pd.concat(parts).drop_duplicates('sentence_uid').reset_index(drop=True)
    final['weak_aspects'] = final['weak_aspects'].astype(str)

    out_path = f'{PROJECT_ROOT}/data/annotation_v2/{slug}_annotation_sample_100.csv'
    final[['sentence_uid', 'reviewId', 'sentence_text', 'weak_aspects']].assign(
        weak_sentiment=final['sentence_text'].apply(weak_sentiment)
    ).to_csv(out_path, index=False)
    print(f'{slug}: quota {quota} -> actual {len(final)} sentences -> {out_path}')

for slug, quota in allocation.items():
    generate_annotation_sample(slug, quota)

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

def review_game(slug):
    labeled_path = f'{PROJECT_ROOT}/data/annotation_v2/{slug}_annotation_sample_LABELED.csv'
    source_path = f'{PROJECT_ROOT}/data/annotation_v2/{slug}_annotation_sample_100.csv'

    if os.path.exists(labeled_path):
        df = pd.read_csv(labeled_path)
        print(f'Resuming review: {labeled_path}')
    else:
        df = pd.read_csv(source_path)
        df['_has_weak_hit'] = df['weak_aspects'].apply(
            lambda x: len(ast.literal_eval(x)) > 0 if isinstance(x, str) else False
        )
        df = df.sort_values('_has_weak_hit', ascending=True).reset_index(drop=True)
        for a in ASPECT_IDS:
            df[f'gold_{a}'] = ''
            df[f'gold_{a}_sentiment'] = ''
        df['_annotated'] = 0
        print(f'Starting review (sentences with no lexicon hit come first): {source_path}')

    total = len(df)
    remaining = df[df['_annotated'] == 0]
    print(f'{slug}: {total} sentences in total, {total - len(remaining)} reviewed, {len(remaining)} remaining')
    if len(remaining) == 0:
        print('🎉 All sentences reviewed!'); return

    state = {'idx': remaining.index[0]}
    progress_out, text_out = widgets.HTML(), widgets.HTML()
    checkboxes = {a: widgets.Checkbox(description=ASPECT_TAXONOMY[a]['label'], value=False,
                                       layout=widgets.Layout(width='420px')) for a in ASPECT_IDS}
    dropdowns = {a: widgets.Dropdown(options=['', 'negative', 'neutral', 'positive'],
                                      layout=widgets.Layout(width='150px')) for a in ASPECT_IDS}
    rows_ui = [widgets.HBox([checkboxes[a], dropdowns[a]]) for a in ASPECT_IDS]

    confirm_btn = widgets.Button(description='✅ Confirm (lexicon suggestion is correct)', button_style='success')
    edited_btn = widgets.Button(description='✏️ Edited, submit', button_style='primary')
    skip_btn = widgets.Button(description='⏭️ No aspect (all zeros)', button_style='info')
    stop_btn = widgets.Button(description='⏸️ Save and pause', button_style='warning')
    output = widgets.Output()

    def render_current():
        idx = state['idx']
        row = df.loc[idx]
        n_done = int(df['_annotated'].sum())
        progress_out.value = f"<b>Progress: {n_done}/{total}</b> | UID: {row['sentence_uid']}"
        text_out.value = f"<h3 style='color:#4CAF50'>Sentence:</h3><p style='font-size:18px'>{row['sentence_text']}</p>"
        weak = ast.literal_eval(row['weak_aspects']) if isinstance(row['weak_aspects'], str) else row.get('weak_aspects', [])
        for a in ASPECT_IDS:
            checkboxes[a].value = a in weak
            dropdowns[a].value = ''

    def save_row(off_topic=False):
        idx = state['idx']
        for a in ASPECT_IDS:
            if off_topic:
                df.at[idx, f'gold_{a}'] = 0
                df.at[idx, f'gold_{a}_sentiment'] = ''
            else:
                present = 1 if checkboxes[a].value else 0
                df.at[idx, f'gold_{a}'] = present
                df.at[idx, f'gold_{a}_sentiment'] = dropdowns[a].value if present else ''
        df.at[idx, '_annotated'] = 1
        df.to_csv(labeled_path, index=False)

    def advance():
        remaining_now = df[df['_annotated'] == 0]
        with output:
            clear_output()
            if len(remaining_now) == 0:
                text_out.value = "<h2 style='color:#4CAF50'>🎉 All sentences reviewed!</h2>"
                for w in rows_ui: w.layout.display = 'none'
                confirm_btn.layout.display = edited_btn.layout.display = skip_btn.layout.display = 'none'
                return
            state['idx'] = remaining_now.index[0]
            render_current()

    confirm_btn.on_click(lambda _: (save_row(False), advance()))
    edited_btn.on_click(lambda _: (save_row(False), advance()))
    skip_btn.on_click(lambda _: (save_row(True), advance()))
    def go_stop(_):
        df.to_csv(labeled_path, index=False)
        with output:
            clear_output()
            print(f'Saved ({int(df["_annotated"].sum())}/{total}); run review_game("{slug}") again to continue')
    stop_btn.on_click(go_stop)

    render_current()
    display(widgets.VBox([progress_out, text_out] + rows_ui +
                          [widgets.HBox([confirm_btn, edited_btn, skip_btn, stop_btn]), output]))

def check_all_progress():
    print(f'royalmatch (legacy reserve set): 1000/1000 done (annotation/ folder, original version, left untouched)')
    for slug in ["royalmatch", "genshin", "mlbb", "pubgm", "arknights"]:
        path = f'{PROJECT_ROOT}/data/annotation_v2/{slug}_annotation_sample_LABELED.csv'
        if os.path.exists(path):
            d = pd.read_csv(path)
            done = int(d.get('_annotated', pd.Series([0])).sum())
            print(f'{slug:12s}: {done}/{len(d)} done (annotation_v2/, new 100-sentence scheme)')
        else:
            print(f'{slug:12s}: not started (annotation_v2/)')

In [ ]:
old_labeled = f'{PROJECT_ROOT}/data/annotation/genshin_annotation_sample_LABELED.csv'
if os.path.exists(old_labeled):
    os.remove(old_labeled)
    print("Removed the record under the old path")

In [ ]:
review_game("genshin")

In [ ]:
review_game("arknights")

In [ ]:
review_game("mlbb")

In [ ]:
review_game("royalmatch")

In [ ]:
review_game("pubgm")

In [ ]:
import pandas as pd
import os

def load_and_validate_gold(slug):
    path = f'{PROJECT_ROOT}/data/annotation_v2/{slug}_annotation_sample_LABELED.csv'
    df = pd.read_csv(path)
    df['game'] = slug

    n_annotated = int(pd.to_numeric(df['_annotated'], errors='coerce').fillna(0).sum()) if '_annotated' in df.columns else len(df)
    print(f'{slug}: {len(df)} sentences, {n_annotated} annotated')

    # Check for aspect=1 with an empty sentiment (a common manual-annotation omission)
    for a in ASPECT_IDS:
        col_a, col_s = f'gold_{a}', f'gold_{a}_sentiment'
        if col_a in df.columns:
            a_num = pd.to_numeric(df[col_a], errors='coerce').fillna(0)
            bad = df[(a_num == 1) & (df[col_s].isna() | (df[col_s] == ''))]
            if len(bad) > 0:
                print(f'  ⚠️ {a}: {len(bad)} sentences have aspect=1 but no sentiment')
    return df

all_gold = []
for slug in ["royalmatch", "genshin", "mlbb", "pubgm", "arknights"]:
    all_gold.append(load_and_validate_gold(slug))

gold_df = pd.concat(all_gold, ignore_index=True)
print(f'\nTotal after merging: {len(gold_df)} sentences')

In [ ]:
for slug in gold_df['game'].unique():
    sub = gold_df[gold_df['game'] == slug]
    counts = {}
    for a in ASPECT_IDS:
        col = f'gold_{a}'
        if col in sub.columns:
            counts[a] = int(pd.to_numeric(sub[col], errors='coerce').fillna(0).sum())
    nonzero = {k: v for k, v in counts.items() if v > 0}
    print(f'{slug}: {nonzero}')

In [ ]:
# Find the Royal Match sentence labelled grind_progression
rm_bad = gold_df[(gold_df['game'] == 'royalmatch') & (pd.to_numeric(gold_df['gold_grind_progression'], errors='coerce') == 1)]
print("=== Royal Match: mislabelled grind_progression ===")
print(rm_bad[['sentence_text', 'gold_grind_progression', 'gold_grind_progression_sentiment']].to_string())

# Find the Arknights sentence labelled matchmaking
ak_bad = gold_df[(gold_df['game'] == 'arknights') & (pd.to_numeric(gold_df['gold_matchmaking'], errors='coerce') == 1)]
print("\n=== Arknights: mislabelled matchmaking ===")
print(ak_bad[['sentence_text', 'gold_matchmaking', 'gold_matchmaking_sentiment']].to_string())

In [ ]:
# 1. Fix the Royal Match sentence: grind_progression -> level_difficulty_balance
mask_rm = (gold_df['game'] == 'royalmatch') & (gold_df['sentence_text'].str.contains('Dragon nest', na=False))
gold_df.loc[mask_rm, 'gold_grind_progression'] = 0
gold_df.loc[mask_rm, 'gold_grind_progression_sentiment'] = ''
gold_df.loc[mask_rm, 'gold_level_difficulty_balance'] = 1
gold_df.loc[mask_rm, 'gold_level_difficulty_balance_sentiment'] = 'negative'

# 2. Fix the Arknights sentence: matchmaking -> account_support
mask_ak = (gold_df['game'] == 'arknights') & (gold_df['sentence_text'].str.contains('fantastic community', na=False))
gold_df.loc[mask_ak, 'gold_matchmaking'] = 0
gold_df.loc[mask_ak, 'gold_matchmaking_sentiment'] = ''
gold_df.loc[mask_ak, 'gold_account_support'] = 1
gold_df.loc[mask_ak, 'gold_account_support_sentiment'] = 'positive'

# Verify there is no genre cross-talk after the fixes
for slug in gold_df['game'].unique():
    sub = gold_df[gold_df['game'] == slug]
    applicable = get_applicable_aspects(slug)
    not_applicable = set(ASPECT_IDS) - applicable
    for a in not_applicable:
        col = f'gold_{a}'
        if col in sub.columns:
            n = int(pd.to_numeric(sub[col], errors='coerce').fillna(0).sum())
            if n > 0:
                print(f'⚠️ {slug} still has {n} sentence(s) labelled with the non-applicable aspect {a}')
print("✅ Check finished; no warning means all fixes are in place")

# Save again
gold_out_path = f'{PROJECT_ROOT}/data/annotation_v2/gold_set_combined.csv'
gold_df.to_csv(gold_out_path, index=False)
print(f'Updated and saved -> {gold_out_path}')

In [ ]:
gold_out_path = f'{PROJECT_ROOT}/data/annotation_v2/gold_set_combined.csv'
gold_df.to_csv(gold_out_path, index=False)
print(f'Saved -> {gold_out_path}: {len(gold_df)} sentences covering {gold_df["game"].nunique()} games')